In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT=Path.cwd().parents[1]
sys.path.insert(0,str(ROOT))

import common as cfg
import helper_funcs as hf

CONFIG=cfg.load_config()
INPUT,OUTPUT,SRC=cfg.task_paths()
OUTPUT.mkdir(parents=True,exist_ok=True)
rng=cfg.initialize(CONFIG['seed'])


In [ ]:
particles=pd.read_csv(INPUT/'h_abc_smc'/'abc_particles.csv')
diagnostics=pd.read_csv(INPUT/'h_abc_smc'/'abc_diagnostics.csv')
raw_path=INPUT/'b_prepare'/'raw_trajectories.csv'


In [ ]:
if not raw_path.exists():
    status=pd.DataFrame([{
        'status':'not_run',
        'reason':'scientist_trajectories.csv was not supplied; aggregate summaries cannot recover individual harvest paths',
        'required_schema':'scientist_id plus columns 0 through 11 with observed zeros and NaN outside follow-up'
    }])
    status.to_csv(OUTPUT/'particle_filter_status.csv',index=False)
    traces=pd.DataFrame()
    scores=pd.DataFrame()
else:
    ids,q=hf.read_trajectory_matrix(raw_path,CONFIG['n_years'])
    model=diagnostics.iloc[0]['model']
    final=particles.loc[(particles['model']==model)&(particles['generation']==particles.loc[particles['model']==model,'generation'].max())]
    sampled=hf.weighted_sample(final,3,rng)
    trace_rows=[]
    score_rows=[]
    for draw,row in sampled.iterrows():
        theta={name:row[name] for name in hf.prior_specs(model)}
        for scientist in range(min(CONFIG['particle_filter_scholars'],len(q))):
            trace,score=hf.abc_particle_filter(q[scientist],model,theta,CONFIG,rng)
            trace['scientist_id']=ids[scientist]
            trace['posterior_draw']=draw
            trace['model']=model
            trace_rows.append(trace)
            score_rows.append({'scientist_id':ids[scientist],'posterior_draw':draw,'model':model,'abc_log_score':score})
    traces=pd.concat(trace_rows,ignore_index=True)
    scores=pd.DataFrame(score_rows)
    traces.to_csv(OUTPUT/'particle_filter_traces.csv',index=False)
    scores.to_csv(OUTPUT/'particle_filter_scores.csv',index=False)
    pd.DataFrame([{'status':'completed','model':model,'scholars':scores['scientist_id'].nunique(),'posterior_draws':scores['posterior_draw'].nunique()}]).to_csv(OUTPUT/'particle_filter_status.csv',index=False)


In [ ]:
if len(traces):
    fig,ax=plt.subplots(figsize=(8,4.5))
    plot=traces.groupby('age',as_index=False)[['observed_h','filtered_h']].mean()
    plot=plot.melt('age',var_name='series',value_name='harvest')
    sns.lineplot(data=plot,x='age',y='harvest',hue='series',style='series',markers=True,ax=ax)
    ax.set(xlabel='career age',ylabel='mean annual harvest',title='ABC particle-filter reconstruction')
    fig.tight_layout()
    fig.savefig(OUTPUT/'particle_filter_reconstruction.png',dpi=220)
    plt.close(fig)
pd.read_csv(OUTPUT/'particle_filter_status.csv')
